## 03 - Hybrid Model
These models predict the instantaneous acceleration of the network. They are evaluated by autoregressively predicting 100 future steps for 100 networks, even though they were trained only on 20-step trajectories from 3 networks.

Because our primary interest lies in the slow modes of the system, we do not focus on the exact position of each individual node (although those predictions are also reasonably accurate). Instead, we evaluate the models using the Poisson ratio as a global mechanical metric and test whether, after 100 autoregressive prediction steps, the simulator correctly reproduces the network’s Poisson ratio

In [ ]:
# Requires: pip install -e .

from course_project.config import ExperimentConfig
from course_project.runner import run_graph_experiment


In [ ]:
import numpy as np
import torch
import pandas as pd
import json
from pathlib import Path
from course_project.data import load_dataset
from course_project.graph import build_graph
from course_project.metrics import evaluate_rollout_pratio_sides
from course_project.models import create_model, resolve_model_inputs
from course_project.utils import resolve_device

force_train = False

full_dataset_path = '../data/2340_dePablo_networks_OOL_undirected.pt'
train_count = 5
val_count = 20

all_sims = load_dataset(full_dataset_path)
train_sims = all_sims[:train_count]
val_sims = all_sims[train_count:train_count + val_count]
test_sims = all_sims[train_count + val_count:]

print('Source dataset:', full_dataset_path)
print('Train sims:', len(train_sims), 'Val sims:', len(val_sims), 'Test sims:', len(test_sims))

time_lag_steps = 0
time_lag_weight = 0

cv_inject_scale_init = 0.6
cv_consistency_weight = 0.6

common_cfg = dict(
    dataset_path=full_dataset_path,
    train_count=train_count,
    val_count=val_count,
    output_root='../results',
    pos_dim=2,
    history=1,
    limit=20,
    hidden_size=64,
    n_layers=2,
    learning_rate=5e-5,
    learning_rate_decay=1,
    weight_decay=0.0,
    epochs=100,
    val_every=10,
    rollout_every=10,
    cv_eval_every=10,
    rollout_steps=100,
    freeze_normalizers_after_epoch=10,
    device='cuda',
)

best_cv_info = json.loads(Path('../results/cv_transformer_best_cv_selection.json').read_text())
best_cv_ckpt = Path(best_cv_info['saved_checkpoint_path'])
print('Using CV checkpoint:', best_cv_ckpt)
print('Best CV fit R2:', float(best_cv_info['best_cv_fit_r2']), 'epoch:', int(best_cv_info['best_cv_epoch']))

repeats = 5
rows = []

def evaluate_selected_checkpoint_on_test(cfg, run_metrics):
    run_dir = Path(cfg.output_root) / cfg.run_name
    final_ckpt = torch.load(run_dir / 'final_checkpoint.pt', map_location='cpu', weights_only=False)
    selected_checkpoint = Path(final_ckpt['selected_checkpoint']['path'])
    cfg_dict = json.loads((run_dir / 'config.json').read_text())
    device = resolve_device(cfg.device)
    init_frames = [test_sims[0][i].to(device) for i in range(cfg.history + 1)]
    init_graph = build_graph(init_frames).to(device)
    
    model = create_model(
        model_type=cfg.model_type,
        init_graph=init_graph,
        pos_dim=cfg.pos_dim,
        hidden_size=cfg.hidden_size,
        n_layers=cfg.n_layers,
        extras=cfg.model_extras,
    ).to(device)
    model.cfg = cfg_dict
    model.load_checkpoint(str(selected_checkpoint))
    model.eval()
    
    model.freeze_normalizers = True
    model_inputs_cls = resolve_model_inputs(cfg.model_type)
    
    with torch.no_grad():
        rollout_metrics = evaluate_rollout_pratio_sides(
            model=model,
            sims=test_sims,
            history=cfg.history,
            rollout_steps=cfg.rollout_steps,
            pos_dim=cfg.pos_dim,
            device=device,
            model_inputs_cls=model_inputs_cls,
        )
    return {
        **run_metrics,
        'selected_checkpoint': str(selected_checkpoint),
        'test_rollout_r2': float(rollout_metrics['rollout_r2']),
        'test_rollout_pearson_r': float(rollout_metrics['rollout_pearson_r']),
        'test_rollout_pos_mse': float(rollout_metrics['rollout_pos_mse']),
    }

def load_or_train_run(cfg):
    run_dir = Path(cfg.output_root) / cfg.run_name
    metrics_path = run_dir / 'metrics.json'
    final_ckpt_path = run_dir / 'final_checkpoint.pt'
    train_stats_path = run_dir / 'train_stats.pt'
    if (not force_train) and metrics_path.exists() and final_ckpt_path.exists() and train_stats_path.exists():
        print('using existing run:', run_dir)
        return json.loads(metrics_path.read_text())
    return run_graph_experiment(cfg)

for i in range(repeats):
    seed = int(torch.randint(0, 2**31 - 1, (1,)).item())
    print('run', i + 1, 'seed:', seed)

    cfg_hybrid = ExperimentConfig(
        run_name=f'hybrid_r{i+1}',
        model_type='hybrid',
        model_extras={
            'num_mlp': 3,
            'cv_checkpoint_path': str(best_cv_ckpt),
            'cv_inject_scale_init': cv_inject_scale_init,
            'cv_consistency_weight': cv_consistency_weight,
            'time_lag_steps': time_lag_steps,
            'time_lag_weight': time_lag_weight,
        },
        **{**common_cfg, 'seed': seed},
    )

    cfg_spatial = ExperimentConfig(
        run_name=f'spatial_baseline_r{i+1}',
        model_type='spatial',
        model_extras={
            'num_mlp': 3,
        },
        **{**common_cfg, 'seed': seed},
    )

    m_hybrid_val = load_or_train_run(cfg_hybrid)
    m_spatial_val = load_or_train_run(cfg_spatial)
    m_hybrid = evaluate_selected_checkpoint_on_test(cfg_hybrid, m_hybrid_val)
    m_spatial = evaluate_selected_checkpoint_on_test(cfg_spatial, m_spatial_val)

    hybrid_ckpt = torch.load(Path(common_cfg['output_root']) / cfg_hybrid.run_name / 'final_checkpoint.pt', map_location='cpu', weights_only=False)
    hybrid_stats = torch.load(Path(common_cfg['output_root']) / cfg_hybrid.run_name / 'train_stats.pt', map_location='cpu', weights_only=False)

    m_hybrid['repeat'] = i + 1
    m_hybrid['cv_scale'] = float(hybrid_ckpt['model_state_dict']['cv_inject_scale'])
    m_hybrid['cv_film_mean'] = float(np.asarray(hybrid_stats['cv_film_mean'], dtype=float)[-1])

    m_spatial['repeat'] = i + 1
    m_spatial['cv_scale'] = np.nan
    m_spatial['cv_film_mean'] = np.nan

    rows.append(m_spatial)
    rows.append(m_hybrid)

runs = pd.DataFrame(rows)

runs_summary = runs[[
    'repeat',
    'run_name',
    'model_type',
    'best_epoch',
    'best_score',
    'rollout_r2',
    'rollout_pearson_r',
    'rollout_pos_mse',
    'test_rollout_r2',
    'test_rollout_pearson_r',
    'test_rollout_pos_mse',
    'cv_scale',
    'cv_film_mean',
]].sort_values(['model_type', 'repeat'])

compare = runs.groupby('model_type', as_index=False).agg(
    mean_val_selected_rollout_r2=('best_score', 'mean'),
    mean_test_rollout_r2=('test_rollout_r2', 'mean'),
    std_test_rollout_r2=('test_rollout_r2', 'std'),
    mean_test_rollout_pos_mse=('test_rollout_pos_mse', 'mean'),
)

print('Per-run results (checkpoint selected on val, final metrics on test):')
runs_summary


Data about the models rollout. Where we autoregressively rollout for 100 steps and measure the poisson ratio against the ground truth trajectory. Higher R2 is better. 

In [ ]:
print('Mean comparison (use this to compare models):')
compare

In [ ]:
# Paper-ready rollout p-ratio scatter plots for the top 2 runs of each model.
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

paper_plot_dir = Path(common_cfg['output_root']) / 'paper_rollout_scatter'
paper_plot_dir.mkdir(parents=True, exist_ok=True)
paper_plot_dpi = 320

def load_selected_model_for_test(row):
    run_dir = Path(common_cfg['output_root']) / row['run_name']
    cfg_dict = json.loads((run_dir / 'config.json').read_text())
    device = resolve_device(cfg_dict['device'])
    init_frames = [test_sims[0][i].to(device) for i in range(cfg_dict['history'] + 1)]
    init_graph = build_graph(init_frames).to(device)
    model = create_model(
        model_type=cfg_dict['model_type'],
        init_graph=init_graph,
        pos_dim=cfg_dict['pos_dim'],
        hidden_size=cfg_dict['hidden_size'],
        n_layers=cfg_dict['n_layers'],
        extras=cfg_dict['model_extras'],
    ).to(device)
    model.cfg = cfg_dict
    model.load_checkpoint(str(row['selected_checkpoint']))
    model.eval()
    model.freeze_normalizers = True
    return model, cfg_dict, device

def collect_rollout_scatter_rows(row):
    model, cfg_dict, device = load_selected_model_for_test(row)
    model_inputs_cls = resolve_model_inputs(cfg_dict['model_type'])
    with torch.no_grad():
        metrics = evaluate_rollout_pratio_sides(
            model=model,
            sims=test_sims,
            history=cfg_dict['history'],
            rollout_steps=cfg_dict['rollout_steps'],
            pos_dim=cfg_dict['pos_dim'],
            device=device,
            model_inputs_cls=model_inputs_cls,
        )
    return pd.DataFrame(metrics['rows'])

def save_paper_scatter(row, panel_label):
    scatter_df = collect_rollout_scatter_rows(row)
    x = scatter_df['target_rollout_sides_p_ratio'].to_numpy(dtype=float)
    y = scatter_df['pred_rollout_p_ratio'].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(4.8, 4.3))
    ax.scatter(x, y, s=28, alpha=0.82, color='#1f77b4', edgecolors='none')

    lo = float(min(x.min(), y.min()))
    hi = float(max(x.max(), y.max()))
    pad = 0.05 * (hi - lo + 1e-8)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], '--', color='black', linewidth=1.0)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Ground-truth final p-ratio')
    ax.set_ylabel('Predicted final p-ratio')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.tick_params(labelsize=9)
    ax.grid(alpha=0.2)
    plt.tight_layout()

    png_path = paper_plot_dir / f'{panel_label}_{row["run_name"]}_rollout_scatter.png'
    pdf_path = paper_plot_dir / f'{panel_label}_{row["run_name"]}_rollout_scatter.pdf'
    fig.savefig(png_path, dpi=paper_plot_dpi, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    print(f'{panel_label}: {row["run_name"]} | test rollout R^2={row["test_rollout_r2"]:.3f}')
    print('saved:', png_path)
    print('saved:', pdf_path)

top_spatial = runs[runs['model_type'] == 'spatial'].sort_values('test_rollout_r2', ascending=False).head(2)
top_hybrid = runs[runs['model_type'] == 'hybrid'].sort_values('test_rollout_r2', ascending=False).head(2)
panel_rows = [top_spatial.iloc[i] for i in range(len(top_spatial))] + [top_hybrid.iloc[i] for i in range(len(top_hybrid))]
panel_labels = ['spatial_top1', 'spatial_top2', 'hybrid_top1', 'hybrid_top2']

for row, panel_label in zip(panel_rows, panel_labels):
    save_paper_scatter(row, panel_label)
